# Gini impurity

Draw two parts from a leaf at random, with replacement. The Gini impurity is the probability that their labels disagree.

If the leaf has class proportions $p_1, \ldots, p_K$,

$$
G = 1 - \sum_{k=1}^{K} p_k^2
$$

The sum is the probability that the two draws match. One minus that sum is the probability that they do not.

For two classes, let $p$ be the proportion of pass. The fail proportion is $1-p$.

$$
\begin{aligned}
G(p)
&= 1 - p^2 - (1-p)^2 \\
&= 1 - p^2 - (1 - 2p + p^2) \\
&= 2p - 2p^2 \\
&= 2p(1-p)
\end{aligned}
$$

$G(0) = 0$ and $G(1) = 0$. A pure leaf has nothing to disagree about. The derivative

$$
G'(p) = 2 - 4p
$$

is zero at $p = 1/2$, and $G''(p) = -4 < 0$, so that critical point is a maximum. The maximum value is

$$
G\left(\frac{1}{2}\right) = 2\cdot\frac{1}{2}\cdot\frac{1}{2} = \frac{1}{2}
$$

For a yes-or-no label, impurity never exceeds $1/2$.


## The eight parts, and the cut that error rate missed

Four pass and four fail:

$$
G = 2\cdot\frac{1}{2}\cdot\frac{1}{2} = \frac{1}{2}
$$

Three pass and one fail:

$$
G = 1 - \left(\frac{3}{4}\right)^2 - \left(\frac{1}{4}\right)^2
= 1 - \frac{9}{16} - \frac{1}{16}
= \frac{6}{16}
= \frac{3}{8}
$$

The cut from the previous lesson splits that leaf into a pure pair and a $1$-and-$1$ pair. The pure side has $G = 0$. The tied side has $G = 1/2$. Weighted by size,

$$
\frac{2}{4}\cdot 0 + \frac{2}{4}\cdot\frac{1}{2} = \frac{1}{4}
$$

The drop is

$$
\frac{3}{8} - \frac{1}{4} = \frac{1}{8}
$$

Error rate dropped by $0$. Gini dropped by $1/8$. That is why the tree will split on Gini, not on the error count.


In [1]:
# Gini from the disagreement formula, then the two-class shortcut.
from fractions import Fraction

def gini(labels):
    n = len(labels)
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    match = sum(Fraction(count, n) ** 2 for count in counts.values())
    return 1 - match

balanced = ["pass"] * 4 + ["fail"] * 4
uneven = ["pass"] * 3 + ["fail"]
assert gini(balanced) == Fraction(1, 2)
assert gini(uneven) == Fraction(3, 8)
assert 2 * Fraction(3, 4) * Fraction(1, 4) == Fraction(3, 8)
weighted = Fraction(2, 4) * gini(["pass", "pass"]) + Fraction(2, 4) * gini(["pass", "fail"])
assert weighted == Fraction(1, 4)
assert gini(uneven) - weighted == Fraction(1, 8)
print("drop", gini(uneven) - weighted)


drop 1/8


## A pitfall

Gini $1/2$ means a balanced yes-or-no leaf. It does not mean "half the parts are errors" in every situation, though on a tie those two numbers match. On three pass and one fail, the error rate of the majority vote is $1/4$, while the Gini value is $3/8$. They are different measurements. Do not subtract one from the other.

## What to carry forward

$G = 1 - \sum p_k^2$, and for two classes this is $2p(1-p)$. It is $0$ on a pure leaf and $1/2$ on a tie. The three-and-one leaf has Gini $3/8$, and the cut that error rate could not see drops that number by $1/8$.
